# Approach B — V3-led closed components with V2 split guards

## Strategy

This version makes **v3's closed-mask components and padded halo the primary proposal mechanism**. It then brings in v2's key protection: before a closed component becomes a single figure, check which original-ink v2 seed groups it contains. If closing would fuse distinct v2 seed groups, the merged closed box is rejected and the component is split back around those seed groups.

For small components outside a closed cluster, v3's padded halo can absorb them, but only if actual ink distance is within the halo distance. If two hosts are similarly plausible, the assignment is rejected unless v2-style containment provides a clear tie-break.

**Best comparison when broken/dashed strokes and external labels are often missed.** This version is intentionally more v3-like and can absorb more nearby labels. Non-seeded standalone proposals use v3's `FIG_FRAC=0.004` output floor by default; set `fig_frac=0` in `find_figures()` if you want to retain all proposal boxes for debugging.

## Parameters worth tuning

- `CLOSE_PX`: stroke-gap support size (at 300 dpi). Start at 5.
- `MIN_AREA`, `SEED_AREA`, and `ATTACH`: inherited from v2; avoid changing all three at once.


In [1]:
"""Approach B — V3-led closed-mask proposals with V2 split/attachment guards.

Morphological closing is the primary way to propose components and boxes (V3's
strength). Before accepting a closed component as one figure, its original-ink
components are checked against V2 seed groups. If a closed cluster contains two
seed groups that V2 regards as distinct, the closed box is NOT allowed to fuse
them. V3's padded halo then absorbs smaller proposals, but V2's actual-ink
nearest-distance competition guards ambiguous assignments.
"""
import cv2 as cv
import numpy as np
from pathlib import Path
import glob

MIN_AREA = 20
SEED_AREA = 3000
TOUCH = 8
NEAR = 45
V2_CONTAIN = 0.5
ATTACH = 48
CLOSE_PX = 5
PAD_PX = 60
CONTAIN = 0.60
FIG_FRAC = 0.004  # V3-style output floor for boxes without a V2 seed; set 0 to disable
MICRO_MIN_AREA = 3
AMBIGUITY_MARGIN = 8


def area(box):
    return max(0, int(box[2] - box[0])) * max(0, int(box[3] - box[1]))


def frac_inside(piece, host):
    pa = area(piece)
    if pa <= 0:
        return 0.0
    iw = min(piece[2], host[2]) - max(piece[0], host[0])
    ih = min(piece[3], host[3]) - max(piece[1], host[1])
    return max(iw, 0) * max(ih, 0) / pa


def frac_inside_padded(piece, host, pad_px):
    halo = [host[0] - pad_px, host[1] - pad_px,
            host[2] + pad_px, host[3] + pad_px]
    return frac_inside(piece, halo)


def _dist_to(mask):
    return cv.distanceTransform(np.where(mask, 0, 1).astype(np.uint8), cv.DIST_L2, 5)


def _union(a, b):
    return [min(int(a[0]), int(b[0])), min(int(a[1]), int(b[1])),
            max(int(a[2]), int(b[2])), max(int(a[3]), int(b[3]))]


def _box_gap(a, b):
    return float(np.hypot(max(a[0] - b[2], b[0] - a[2], 0),
                          max(a[1] - b[3], b[1] - a[3], 0)))


def _closed_map(ink, labels, n, close_px):
    if close_px > 1:
        kernel = cv.getStructuringElement(cv.MORPH_RECT, (int(close_px), int(close_px)))
        closed = cv.morphologyEx(ink, cv.MORPH_CLOSE, kernel)
    else:
        closed = ink.copy()
    n_closed, closed_labels, closed_stats, _ = cv.connectedComponentsWithStats(closed, connectivity=8)
    common = (labels > 0) & (closed_labels > 0)
    component_cluster, cluster_components = {}, {}
    if np.any(common):
        pairs = np.column_stack((closed_labels[common], labels[common]))
        unique_pairs, counts = np.unique(pairs, axis=0, return_counts=True)
        best = {}
        for (cid, oid), count in zip(unique_pairs, counts):
            cid, oid = int(cid), int(oid)
            if oid not in best or count > best[oid][1]:
                best[oid] = (cid, int(count))
        component_cluster = {oid: pair[0] for oid, pair in best.items()}
        for oid, cid in component_cluster.items():
            cluster_components.setdefault(cid, set()).add(oid)
    return closed, closed_labels, closed_stats, component_cluster, cluster_components


def find_figures(ink, min_area=MIN_AREA, seed_area=SEED_AREA, touch=TOUCH,
                 near=NEAR, attach=ATTACH, dpi=300, close_px=CLOSE_PX,
                 pad_px=PAD_PX, contain=CONTAIN, fig_frac=FIG_FRAC):
    """Return boxes [x1,y1,x2,y2] with exclusive x2/y2.

    This deliberately uses V3-style closed components and halo absorption as the
    proposal mechanism, while V2 seed grouping is a hard split guard.
    """
    if ink is None or ink.ndim != 2:
        raise ValueError("ink must be a 2-D grayscale/binary image")
    if dpi <= 0:
        raise ValueError("dpi must be positive")
    ink = np.where(ink > 0, 255, 0).astype(np.uint8)
    H, W = ink.shape
    s = float(dpi) / 300.0
    min_area_px, seed_area_px = min_area * s * s, seed_area * s * s
    touch_px, near_px, attach_px, pad_px_scaled = touch * s, near * s, attach * s, pad_px * s
    ambiguity_px = AMBIGUITY_MARGIN * s

    n, labels, stats, _ = cv.connectedComponentsWithStats(ink, connectivity=8)
    x, y, w, h, component_area = (stats[:, k].astype(np.int64) for k in range(5))
    boxes = np.stack([x, y, x + w, y + h], axis=1)
    fill = component_area / np.maximum(w * h, 1)
    aspect = np.maximum(w, h) / np.maximum(np.minimum(w, h), 1)
    big = component_area >= seed_area_px
    junk = big & (((aspect >= 15) & (np.minimum(w, h) < 0.03 * W)) | (fill > 0.8))
    base_keep = (component_area >= min_area_px) & ~junk
    base_keep[0] = False

    closed, closed_labels, closed_stats, comp_cluster, cluster_components = _closed_map(
        ink, labels, n, int(round(close_px * s)))

    # ----- V2 hard seed grouping on original ink ------------------------------------
    seeds = [int(i) for i in np.argsort(-component_area) if base_keep[i] and big[i]]
    seed_ids = set(seeds)
    seed_groups = [dict(ids=[i], seed_ids={i}, box=list(map(int, boxes[i])),
                        mass=int(component_area[i]), has_seed=True)
                   for i in seeds]

    def group_mask(group, roi):
        return np.isin(labels[roi[1]:roi[3], roi[0]:roi[2]], group["ids"])

    def ink_gap(g1, g2):
        if _box_gap(g1["box"], g2["box"]) > near_px:
            return np.inf
        roi = _union(g1["box"], g2["box"])
        m1, m2 = group_mask(g1, roi), group_mask(g2, roi)
        if not m1.any() or not m2.any():
            return np.inf
        return float(_dist_to(m1)[m2].min())

    merged = True
    while merged:
        merged = False
        for p in range(len(seed_groups)):
            for q in range(p + 1, len(seed_groups)):
                host, small = sorted((seed_groups[p], seed_groups[q]), key=lambda g: -g["mass"])
                gap = ink_gap(host, small)
                # V2's guarded rule: never merge solely because boxes overlap.
                if gap <= touch_px or (gap <= near_px and frac_inside(small["box"], host["box"]) >= V2_CONTAIN):
                    host["ids"] += small["ids"]
                    host["seed_ids"] |= small["seed_ids"]
                    host["box"] = _union(host["box"], small["box"])
                    host["mass"] += small["mass"]
                    seed_groups.remove(small)
                    merged = True
                    break
            if merged:
                break

    seed_group_of = {sid: gi for gi, g in enumerate(seed_groups) for sid in g["seed_ids"]}
    seeded_component_ids = set(seed_ids)
    free_proposals = []
    handled_ids = set(seed_ids)
    close_scale = int(round(close_px * s))

    def dist_component_to_group(cid, group, max_dist):
        # Crop only around the candidate. This avoids making an enormous ROI for
        # stitched pages while still measuring true original-ink distance.
        bx = boxes[cid]
        pad = int(np.ceil(max_dist)) + 2
        x1, y1 = max(0, int(bx[0]) - pad), max(0, int(bx[1]) - pad)
        x2, y2 = min(W, int(bx[2]) + pad), min(H, int(bx[3]) + pad)
        roi_labels = labels[y1:y2, x1:x2]
        host_mask = np.isin(roi_labels, group["ids"])
        own_mask = roi_labels == cid
        if not host_mask.any() or not own_mask.any():
            return np.inf
        d = float(_dist_to(host_mask)[own_mask].min())
        return d if d <= max_dist else np.inf

    # ----- V3 closed components are primary proposals --------------------------------
    for cid in sorted(cluster_components):
        members = cluster_components[cid]
        valid_members = [i for i in members if 0 < i < n and not junk[i]
                         and component_area[i] >= MICRO_MIN_AREA]
        if not valid_members:
            continue
        groups_here = sorted({seed_group_of[sid] for sid in members if sid in seed_group_of})
        closed_box = [int(closed_stats[cid, cv.CC_STAT_LEFT]),
                      int(closed_stats[cid, cv.CC_STAT_TOP]),
                      int(closed_stats[cid, cv.CC_STAT_LEFT] + closed_stats[cid, cv.CC_STAT_WIDTH]),
                      int(closed_stats[cid, cv.CC_STAT_TOP] + closed_stats[cid, cv.CC_STAT_HEIGHT])]
        closed_area = int(closed_stats[cid, cv.CC_STAT_AREA])

        if len(groups_here) == 1:
            # A closed cluster with one V2 seed group is safe to use as a V3 box
            # proposal. Include fine fragments in this cluster, but not ignored junk.
            host = seed_groups[groups_here[0]]
            host["box"] = _union(host["box"], closed_box)
            for i in valid_members:
                if i not in host["ids"]:
                    host["ids"].append(i)
                    host["mass"] += int(component_area[i])
                handled_ids.add(i)
            continue

        if len(groups_here) > 1:
            # Closing joined distinct V2 figures. Do NOT use the joined closed box.
            # Non-seed fragments may go to one group only when distance is decisive.
            candidate_groups = [seed_groups[gi] for gi in groups_here]
            for i in valid_members:
                if i in seed_ids:
                    handled_ids.add(i)
                    continue
                distances = [(dist_component_to_group(i, g, max(attach_px, pad_px_scaled)), gi)
                            for gi, g in zip(groups_here, candidate_groups)]
                distances = sorted((d, gi) for d, gi in distances if np.isfinite(d))
                assigned = False
                if distances:
                    d1, gi1 = distances[0]
                    d2 = distances[1][0] if len(distances) > 1 else np.inf
                    host = seed_groups[gi1]
                    if (d1 <= attach_px and
                        ((d2 - d1) >= ambiguity_px or
                         (d1 <= near_px and frac_inside(boxes[i], host["box"]) >= V2_CONTAIN))):
                        host["ids"].append(i)
                        host["mass"] += int(component_area[i])
                        host["box"] = _union(host["box"], boxes[i])
                        handled_ids.add(i)
                        assigned = True
                if not assigned and (component_area[i] >= min_area_px or closed_area >= min_area_px):
                    free_proposals.append(dict(ids=[i], seed_ids=set(), box=list(map(int, boxes[i])),
                                               mass=int(component_area[i]), has_seed=False))
                    handled_ids.add(i)
            continue

        # No seed in this closed cluster: preserve V3's closed-mask box as a proposal.
        mass = int(sum(component_area[i] for i in valid_members))
        if max(closed_area, mass) < min_area_px:
            continue
        free_proposals.append(dict(ids=list(valid_members), seed_ids=set(), box=closed_box,
                                   mass=closed_area, has_seed=False))
        handled_ids.update(valid_members)

    # Components not mapped by a closing cluster get original-ink proposals.
    for i in range(1, n):
        if i in handled_ids or not base_keep[i] or junk[i] or i in seed_ids:
            continue
        free_proposals.append(dict(ids=[i], seed_ids=set(), box=list(map(int, boxes[i])),
                                   mass=int(component_area[i]), has_seed=False))
        handled_ids.add(i)

    # ----- V3 halo absorption guarded by V2 ink distances -----------------------------
    result = list(seed_groups)  # Seed groups are immutable hosts for one another.
    max_distance = max(attach_px, pad_px_scaled)
    for piece in sorted(free_proposals, key=lambda g: -g["mass"]):
        candidates = []
        for hi, host in enumerate(result):
            # A seeded figure cannot be absorbed into a non-seeded host, and distinct
            # V2 seed groups are never halo-merged together.
            if piece["has_seed"] or (piece["seed_ids"] and host["seed_ids"]):
                continue
            if piece["mass"] > host["mass"] * 0.25:
                continue
            halo_overlap = frac_inside_padded(piece["box"], host["box"], pad_px_scaled)
            if halo_overlap < contain:
                continue
            d = dist_component_to_group(piece["ids"][0], host, max_distance) if len(piece["ids"]) == 1 else np.inf
            if len(piece["ids"]) > 1:
                # Distance for cluster proposals with several original labels.
                ds = [dist_component_to_group(i, host, max_distance) for i in piece["ids"]]
                ds = [v for v in ds if np.isfinite(v)]
                d = min(ds) if ds else np.inf
            if np.isfinite(d) and d <= max_distance:
                candidates.append((d, -halo_overlap, hi))
        candidates.sort()
        chosen = None
        if len(candidates) == 1:
            chosen = candidates[0][2]
        elif len(candidates) > 1:
            d1, _, h1 = candidates[0]
            d2 = candidates[1][0]
            host = result[h1]
            # If distances tie, only use V2's direct containment as a tie-breaker.
            if d2 - d1 >= ambiguity_px or (d1 <= near_px and frac_inside(piece["box"], host["box"]) >= V2_CONTAIN):
                chosen = h1
        if chosen is None:
            result.append(piece)
        else:
            host = result[chosen]
            host["box"] = _union(host["box"], piece["box"])
            host["ids"] = list(dict.fromkeys(host["ids"] + piece["ids"]))
            host["mass"] += int(piece["mass"])

    # V3's global page-area filter is used only for proposals without V2 seeds.
    min_figure_box_area = fig_frac * W * H
    final_groups = [g for g in result if g["has_seed"] or area(g["box"]) >= min_figure_box_area]
    return [[int(v) for v in g["box"]] for g in final_groups]


def reading_order(boxes):
    rows = []
    for b in sorted(boxes, key=lambda b: b[1]):
        for row in rows:
            top, bottom = min(c[1] for c in row), max(c[3] for c in row)
            if min(bottom, b[3]) - max(top, b[1]) >= 0.5 * min(bottom - top, b[3] - b[1]):
                row.append(b)
                break
        else:
            rows.append([b])
    return [b for row in rows for b in sorted(row, key=lambda b: b[0])]


def extract_image(src, debug_out=None, dpi=300, close_px=CLOSE_PX):
    gray = cv.imread(str(src), cv.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f"Could not read image: {src}")
    _, ink = cv.threshold(gray, 128, 255, cv.THRESH_BINARY_INV)
    figures = reading_order(find_figures(ink, dpi=dpi, close_px=close_px))
    if debug_out is not None:
        debug = cv.cvtColor(ink, cv.COLOR_GRAY2BGR)
        for i, (x1, y1, x2, y2) in enumerate(figures, 1):
            cv.rectangle(debug, (x1, y1), (x2 - 1, y2 - 1), (0, 255, 0), 2)
            cv.putText(debug, str(i), (x1 + 8, y1 + 40), cv.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3)
        out = Path(debug_out)
        out.parent.mkdir(parents=True, exist_ok=True)
        if not cv.imwrite(str(out), debug):
            raise IOError(f"Could not write debug overlay: {out}")
    return figures


def run_folder(input_dir="../../data/figure_extractor_test/",
               output_dir="../../results/figure_extractor_B_v3_led/"):
    paths = sorted(glob.glob(str(Path(input_dir) / "*.png")))
    if not paths:
        print(f"No PNG files found in {input_dir!r}; update input_dir to your image folder.")
        return
    outdir = Path(output_dir)
    outdir.mkdir(parents=True, exist_ok=True)
    for src in paths:
        out = outdir / f"debug_{Path(src).name}"
        boxes = extract_image(src, out)
        print(f"{Path(src).name}: {len(boxes)} figures -> {out}")



## Synthetic smoke test

This test checks two things: the split guard preserves separated/nested seeded figures even though v3-style closing is central, and the halo can attach a small external label that is just beyond v2's default `ATTACH=48` but inside v3's `PAD_PX=60`. These are synthetic examples only.

In [2]:
_demo = np.zeros((650, 900), dtype=np.uint8)
cv.rectangle(_demo, (60, 70), (205, 215), 255, 8)
cv.rectangle(_demo, (310, 70), (455, 215), 255, 8)
cv.line(_demo, (105, 63), (110, 63), 255, 1)
cv.rectangle(_demo, (570, 60), (850, 340), 255, 8)
cv.rectangle(_demo, (660, 150), (760, 250), 255, 8)
_demo_boxes = reading_order(find_figures(_demo, close_px=5))
assert len(_demo_boxes) == 4, f"Expected 4 separated/nested seeded figures, got {len(_demo_boxes)}"

# A small external component ~55 px from a figure: v2's ATTACH would not claim it,
# while v3's padded halo (60 px) may absorb it when the guarded conditions pass.
_halo_demo = np.zeros((350, 450), dtype=np.uint8)
cv.rectangle(_halo_demo, (50, 60), (205, 215), 255, 8)
cv.rectangle(_halo_demo, (260, 98), (267, 105), 255, 1)
_halo_boxes = find_figures(_halo_demo, close_px=0)
assert len(_halo_boxes) == 1 and _halo_boxes[0][2] > 250, f"Expected halo to attach the external label, got {_halo_boxes}"
print("PASS: v2 seed split guard preserved four figures; v3 halo absorbed the external label.")
print("V3-led boxes:", _demo_boxes)
print("Halo demonstration boxes:", _halo_boxes)

PASS: v2 seed split guard preserved four figures; v3 halo absorbed the external label.
V3-led boxes: [[56, 63, 210, 220], [306, 66, 460, 220], [566, 56, 855, 345], [656, 146, 765, 255]]
Halo demonstration boxes: [[46, 56, 268, 220]]


## Run on your image folder

Update the input path if your notebook's working directory differs. Outputs go to a **new, approach-specific folder** so the original v2/v3 overlays are not overwritten. If the path does not contain PNG files, the runner prints a message and exits without failing.

In [3]:
run_folder("../../data/figure_extractor_test/", "../../results/figure_extractor_B_v3_led/")

2382_2387_page_2.png: 3 figures -> ../../results/figure_extractor_B_v3_led/debug_2382_2387_page_2.png
2393_2398_page_3.png: 3 figures -> ../../results/figure_extractor_B_v3_led/debug_2393_2398_page_3.png
2394_2399_page_2.png: 2 figures -> ../../results/figure_extractor_B_v3_led/debug_2394_2399_page_2.png
2398_2403_page_3.png: 3 figures -> ../../results/figure_extractor_B_v3_led/debug_2398_2403_page_3.png
2398_2403_page_4.png: 3 figures -> ../../results/figure_extractor_B_v3_led/debug_2398_2403_page_4.png
2424_2429_page_2.png: 2 figures -> ../../results/figure_extractor_B_v3_led/debug_2424_2429_page_2.png
2424_2429_page_3.png: 3 figures -> ../../results/figure_extractor_B_v3_led/debug_2424_2429_page_3.png
2451_2456_page_2.png: 6 figures -> ../../results/figure_extractor_B_v3_led/debug_2451_2456_page_2.png
2451_2456_page_3.png: 9 figures -> ../../results/figure_extractor_B_v3_led/debug_2451_2456_page_3.png
2453_2458_page_3.png: 8 figures -> ../../results/figure_extractor_B_v3_led/debug_2